In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/processed_stage1.csv")  # put correct path

df.head()

,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,...,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,hardship_flag,disbursement_method,debt_settlement_flag,target,target1
0,9600.0,36 months,12.99,323.42,C,C1,NaN,NaN,RENT,21900.0,...,0.0,11600.0,4509.0,2400.0,0.0,N,Cash,N,0,0
1,4000.0,36 months,6.68,122.93,A,A3,System Analyst,4 years,MORTGAGE,83000.0,...,0.0,222616.0,64253.0,5600.0,76154.0,N,Cash,N,0,0
2,6025.0,36 months,10.91,197.00,B,B4,Admin assistant,10+ years,RENT,52000.0,...,0.0,32227.0,5559.0,11000.0,11127.0,N,Cash,N,0,0
3,20000.0,36 months,9.49,640.57,B,B2,Manager,10+ years,MORTGAGE,100000.0,...,0.0,322295.0,33695.0,23900.0,29995.0,N,Cash,N,0,0
4,1000.0,36 months,8.18,31.42,B,B1,NaN,NaN,RENT,23000.0,...,0.0,9979.0,7252.0,1700.0,5779.0,N,Cash,N,0,0


In [2]:
df.shape

(59494, 79)

Day 3 — Data Cleaning (Execution Guide)
🎯 Goal

Convert raw messy dataset → clean, model-ready dataset

🔥 Step 1: Drop Useless Columns

👉 Rule:

Drop columns with >40% missing values
Drop IDs / irrelevant text fields

In [3]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

missing_percentage = (missing / len(df)) * 100

pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_percentage
})

,Missing Count,Missing %
mths_since_recent_inq,7592,12.760951
num_tl_120dpd_2m,5170,8.689952
mo_sin_old_il_acct,4535,7.622617
emp_title,3834,6.444347
emp_length,3470,5.832521
pct_tl_nvr_dlq,2895,4.866037
num_actv_rev_tl,2890,4.857633
total_rev_hi_lim,2890,4.857633
avg_cur_bal,2890,4.857633
tot_cur_bal,2890,4.857633


In [4]:
missing_percent = df.isnull().mean() * 100
cols_to_drop = missing_percent[missing_percent > 40].index
df = df.drop(columns=cols_to_drop)

In [5]:
df.shape

(59494, 79)

In [6]:
df.isnull().sum().sum() #we need to handle these values as these many are NULL

np.int64(108858)

Step 2: Handle Missing Values
📊 Numerical Columns

Fill with median (robust to outliers)

In [10]:
num_cols = df.select_dtypes(include=['int64','float64']).columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

🏷️ Categorical Columns

Fill with mode

In [12]:
cat_cols = df.select_dtypes(include=['object']).columns

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

C:\Users\A\AppData\Local\Temp\ipykernel_12620\3477175695.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns


In [13]:
df.isnull().sum().sum()  #now we do not have NULL

np.int64(0)

🧠 Pro Insight (Interview Level)

If interviewer asks:

👉 “Why avoid inplace?”

You say:

“Because pandas uses copy-on-write internally, inplace operations on chained indexing don’t guarantee modification of the original dataframe. Explicit assignment is safer and more predictable.”

🔥 This answer = strong signal

Step 3: Convert Data Types

In [ ]:
# Step 3: Convert Data Types
# 📅 Date Columns

# Example:

# issue_d
# earliest_cr_line
# df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y')
# df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y')

# 👉 Why important?

# Later → create time-based features (VERY IMPORTANT in fintech)
# 🏷️ Employment Length Conversion

# Example values:

# “10+ years”
# “< 1 year”

# 👉 Convert to numeric:

# def emp_to_num(x):
#     if pd.isnull(x):
#         return 0
#     if x == '< 1 year':
#         return 0
#     if x == '10+ years':
#         return 10
#     return int(x.split()[0])

# df['emp_length'] = df['emp_length'].apply(emp_to_num)

In [18]:
num_cols = df.select_dtypes(include=['int64','float64']).columns
print(num_cols,"\n")
cat_cols = df.select_dtypes(include=['object']).columns
print(cat_cols,"\n")
 

Index(['loan_amnt', 'int_rate', 'installment', 'annual_inc', 'dti',
       'delinq_2yrs', 'fico_range_low', 'fico_range_high', 'inq_last_6mths',
       'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
       'out_prncp', 'out_prncp_inv', 'last_fico_range_high',
       'last_fico_range_low', 'collections_12_mths_ex_med', 'acc_now_delinq',
       'tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim',
       'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util',
       'chargeoff_within_12_mths', 'delinq_amnt', 'mo_sin_old_il_acct',
       'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl',
       'mort_acc', 'mths_since_recent_bc', 'mths_since_recent_inq',
       'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl',
       'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl',
       'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_120dpd_2m',
       'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m',
       'pct_tl_nvr_

C:\Users\A\AppData\Local\Temp\ipykernel_12620\3757979821.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns


In [20]:
df['term'].unique()

<StringArray>
[' 36 months', ' 60 months']
Length: 2, dtype: str

In [24]:
df['term'] = df['term'].str.replace('months', '')
df['term'] = df['term'].str.strip()
df['term'] = df['term'].astype(int)
df['term'].unique()

array([36, 60])

In [ ]:
# df['term'] = df['term'].astype(str)


In [25]:
df['grade'].unique()  #not converted to numeric as of now

<StringArray>
['C', 'A', 'B', 'D', 'E', 'F', 'G']
Length: 7, dtype: str

In [29]:
columns_unique = ['term', 'grade', 'sub_grade', 'emp_title', 'emp_length',
       'home_ownership', 'verification_status', 'issue_d', 'loan_status',
       'pymnt_plan', 'purpose', 'addr_state', 'earliest_cr_line',
       'initial_list_status', 'last_credit_pull_d', 'application_type',
       'hardship_flag', 'disbursement_method', 'debt_settlement_flag']
for col in df[columns_unique]:
    print(f"Column {col}:", df[col].unique())




Column term: [36 60]
Column grade: <StringArray>
['C', 'A', 'B', 'D', 'E', 'F', 'G']
Length: 7, dtype: str
Column sub_grade: <StringArray>
['C1', 'A3', 'B4', 'B2', 'B1', 'C4', 'A2', 'B3', 'C2', 'D4', 'E5', 'F3', 'C5',
 'C3', 'A1', 'E3', 'B5', 'G3', 'E4', 'A5', 'E2', 'A4', 'D5', 'D3', 'D1', 'E1',
 'D2', 'G1', 'F2', 'F4', 'F1', 'G2', 'G4', 'F5', 'G5']
Length: 35, dtype: str
Column emp_title: <StringArray>
[                                'Teacher',
                          'System Analyst',
                         'Admin assistant',
                                 'Manager',
                                 'Casting',
                     'Equipment operator ',
              'Client Solutions Executive',
                                    'HVAC',
                        'Product Manager ',
                               'machinist',
 ...
                                     'otm',
 'Executive Assistant to the Mayor & Mngr',
  'Product Quality Assurance Technologist',
                

In [ ]:
# emp_length
# issue_d
# earliest_cr_line
# last_credit_pull_d

# emp_length

In [31]:
df['emp_length'].unique()

<StringArray>
['10+ years',   '4 years',   '5 years',   '8 years',   '7 years',   '2 years',
   '6 years',    '1 year',   '3 years',   '9 years',  '< 1 year']
Length: 11, dtype: str

In [32]:
df['emp_length'] = df['emp_length'].str.replace('+', '', regex=False)
df['emp_length'] = df['emp_length'].str.replace('years', '', regex=False)
df['emp_length'] = df['emp_length'].str.replace('year', '', regex=False)
df['emp_length'] = df['emp_length'].str.replace('< 1', '0', regex=False)

df['emp_length'] = df['emp_length'].str.strip()

df['emp_length'] = pd.to_numeric(df['emp_length'], errors='coerce')
df['emp_length'].unique()

array([10,  4,  5,  8,  7,  2,  6,  1,  3,  9,  0])

In [33]:
df['emp_length'].head()


0    10
1     4
2    10
3    10
4    10
Name: emp_length, dtype: int64

In [37]:
df['emp_length'].isnull().sum()

np.int64(0)

In [36]:
# 💡 Handle missing values:
df['emp_length'] = df['emp_length'].fillna(df['emp_length'].median())

# issue_d

In [38]:
df.select_dtypes(include='object').columns

C:\Users\A\AppData\Local\Temp\ipykernel_12620\3732952691.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.select_dtypes(include='object').columns


Index(['grade', 'sub_grade', 'emp_title', 'home_ownership',
       'verification_status', 'issue_d', 'loan_status', 'pymnt_plan',
       'purpose', 'addr_state', 'earliest_cr_line', 'initial_list_status',
       'last_credit_pull_d', 'application_type', 'hardship_flag',
       'disbursement_method', 'debt_settlement_flag'],
      dtype='str')

In [ ]:
df['earliest_cr_line'].unique()
# df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'])

# df['credit_age'] = (pd.to_datetime('today') - df['earliest_cr_line']).dt.days // 365

<StringArray>
['Apr-2001', 'Sep-2003', 'Jun-2005', 'Aug-1990', 'Sep-1986', 'Feb-2007',
 'Mar-2006', 'Sep-2005', 'Mar-2003', 'Sep-2000',
 ...
 'Aug-1964', 'Jul-1966', 'Aug-1963', 'Sep-1971', 'Oct-1959', 'Oct-1960',
 'Jun-1963', 'Sep-1969', 'Dec-1962', 'Nov-1959']
Length: 631, dtype: str

In [46]:
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y')
df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y')

In [47]:
df['last_credit_pull_d'] = pd.to_datetime(df['last_credit_pull_d'], format='%b-%Y')



In [48]:
df['earliest_cr_line'].unique()


<DatetimeArray>
['2001-04-01 00:00:00', '2003-09-01 00:00:00', '2005-06-01 00:00:00',
 '1990-08-01 00:00:00', '1986-09-01 00:00:00', '2007-02-01 00:00:00',
 '2006-03-01 00:00:00', '2005-09-01 00:00:00', '2003-03-01 00:00:00',
 '2000-09-01 00:00:00',
 ...
 '1964-08-01 00:00:00', '1966-07-01 00:00:00', '1963-08-01 00:00:00',
 '1971-09-01 00:00:00', '1959-10-01 00:00:00', '1960-10-01 00:00:00',
 '1963-06-01 00:00:00', '1969-09-01 00:00:00', '1962-12-01 00:00:00',
 '1959-11-01 00:00:00']
Length: 631, dtype: datetime64[us]

🔥 Pro Tip (Interview Level)

You can say:

“I converted semi-structured categorical columns like emp_length and term into numeric features to preserve ordinal information and improve model performance.”

In [49]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 59494 entries, 0 to 59493
Data columns (total 79 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   loan_amnt                   59494 non-null  float64       
 1   term                        59494 non-null  int64         
 2   int_rate                    59494 non-null  float64       
 3   installment                 59494 non-null  float64       
 4   grade                       59494 non-null  str           
 5   sub_grade                   59494 non-null  str           
 6   emp_title                   59494 non-null  str           
 7   emp_length                  59494 non-null  int64         
 8   home_ownership              59494 non-null  str           
 9   annual_inc                  59494 non-null  float64       
 10  verification_status         59494 non-null  str           
 11  issue_d                     59494 non-null  datetime64[us]
 12  l

In [50]:
df.to_csv("../data/processed/processed_stage2.csv", index=False)

In [ ]:
# Next Notebook

# ⚠️ Step 4: Handle Outliers (Important)

# Focus on:

# annual_inc
# loan_amnt
# dti

# 👉 Use capping (IQR method)

# def cap_outliers(col):
#     Q1 = df[col].quantile(0.25)
#     Q3 = df[col].quantile(0.75)
#     IQR = Q3 - Q1
    
#     lower = Q1 - 1.5 * IQR
#     upper = Q3 + 1.5 * IQR
    
#     df[col] = df[col].clip(lower, upper)

# for col in ['annual_inc','loan_amnt','dti']:
#     cap_outliers(col)
# 🧠 Step 5: Encode Basic Categories (Light Encoding Only)

# 👉 Don’t go heavy yet (Day 4 is for that)

# For now:

# df = pd.get_dummies(df, drop_first=True)